# Build a RAG Pipeline

In the last lecture, we saw how Retrieval-Augmented Generation (RAG) lets an LLM answer questions using information it wasn’t trained on. In this activity, you’ll build a RAG pipeline from scratch over the [SCP Foundation Wiki](https://scp-wiki.wikidot.com/) — a collaborative fiction project where contributors write pseudo-scientific containment documents about fictional anomalous entities.

## Setup

Run the cells below to load the models you’ll need. We’re using `all-MiniLM-L6-v2` for embeddings and `Qwen2.5-0.5B-Instruct` for generation.

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

gen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name, dtype=torch.float16)

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)
print(f"Generation model on {device}")

## Task 1: Explore the Data

Load the SCP Wiki dataset from HuggingFace and explore it. Each entry has a `description` (short summary) and `content` (full article text). Look at a few entries, check the length distribution, and note the structural variation — some entries are short, others are very long, and they contain a mix of structured headers, prose, interview transcripts, and redacted text.

In [ ]:
# Combine both splits into one dataset
dataset = load_dataset("PhilSad/SCP-Wiki-Dataset", split="train+test")
df = pd.DataFrame(dataset)
print(f"{len(df)} entries loaded")
df.head()

In [ ]:
# Explore the dataset. Look at a few entries, check the distribution of
# content lengths, and get a feel for the data.
# YOUR CODE HERE

*What challenges do you see for embedding these documents?*

## Task 2: Chunk the Corpus

The embedding model has a fixed maximum input length. Documents that exceed it will be truncated, losing information. Write a function that takes the dataset and returns a list of text chunks suitable for embedding.

There is no single right answer here — think about the structure of the data you just explored and decide on a strategy.

In [ ]:
def chunk_corpus(df):
    """Take the SCP dataframe and return a list of text chunks for embedding."""
    # YOUR CODE HERE
    pass

In [ ]:
chunks = chunk_corpus(df)
print(f"{len(chunks)} chunks created")

*What trade-offs did you make? What might break?*

## Task 3: Embed the Chunks

Use the `SentenceTransformer` model to encode your chunks into embeddings.

In [ ]:
# Encode your chunks using emb_model. Store the result as chunk_embeddings.
# YOUR CODE HERE

## Task 4: Write Retrieval

Write a function that takes a query, embeds it, computes cosine similarity against all chunk embeddings, and returns the top-k most similar chunks with their scores.

In [ ]:
def retrieve(query, chunk_embeddings, chunks, emb_model, k=3):
    """Return the top-k most similar chunks to the query.
    
    Each result should be a dict with 'text' and 'score' keys.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test your retrieval function.
results = retrieve("What SCP is a staircase?", chunk_embeddings, chunks, emb_model)

for r in results:
    print(f"[{r['score']:.3f}]")
    print(r["text"][:200])
    print()

In [ ]:
# Try a few queries of your own.
# YOUR CODE HERE

## Task 5: Build the RAG Pipeline

Write a function that:
1. Retrieves relevant chunks for a question
2. Constructs a prompt that includes the retrieved context
3. Generates an answer using the language model

You’ll need to write the prompt template yourself — think about what instructions will help the model use the context effectively.

In [ ]:
def ask(question, chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer, k=3):
    """Retrieve context and generate an answer to the question."""
    # YOUR CODE HERE
    pass

In [ ]:
# Test your RAG pipeline.
print(ask("What SCP is a staircase that goes down forever?", chunk_embeddings, chunks, emb_model, gen_model, gen_tokenizer))

In [ ]:
# Compare: ask the model the same question WITHOUT retrieval context.
question = "What SCP is a staircase that goes down forever?"

messages = [{"role": "user", "content": question}]
inputs = gen_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)

gen_model.eval()
with torch.no_grad():
    output = gen_model.generate(**inputs, max_new_tokens=256, do_sample=False)

print(gen_tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
# Try more queries. Find cases where RAG helps and cases where it doesn't.
# YOUR CODE HERE

## Task 6: Reflection

- Find a query where RAG helps and one where it doesn’t. Why?
- How did your chunking strategy affect retrieval quality?